# Bag-of-words, TF-IDF, Vector Space Model (VSM), and Latent Semantic Analysis

* Bag-of-words representation of documents
* TF-IDF weighting
* Vector Space Model (VSM) and cosine similarity
* Latent Semantic Analysis

In [2]:
import sys
print(sys.version)

3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]


In [3]:
import numpy as np
print(np.__version__)

1.26.4


In [4]:
# !pip install scikit-learn
# !pip install "numpy<2"

In [5]:
!pip install pandas

In [6]:
!pip install gensim

In [7]:
text = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?',
]

## TF-IDF with scikit-learn

**tf-idf = term_frequency * inverse_document_frequency**
**term_frequency** = number of times a given term appears in document
**inverse_document_frequency** = log(total number of documents / number of documents with term) + 1**\***

*There are a bunch of slightly different ways that you can calculate inverse document frequency. The version of idf that we're going to use is the scikit-learn default, which uses "smoothing" aka it adds a "1" to the numerator and denominator:

**inverse_document_frequency**  = log((1 + total_number_of_documents) / (number_of_documents_with_term +1)) + 1

<div class="margin sidebar" style=" padding: 10px">

> If smooth_idf=True (the default), the constant “1” is added to the numerator and denominator of the idf as if an extra document was seen containing every term in the collection exactly once, which prevents zero divisions: idf(t) = log [ (1 + n) / (1 + df(t)) ] + 1.  
> -[scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer)

</div>

-- edited upon Melanie Walsh, Introduction to Cultural Analytics & Python.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

To calculate tf–idf scores for every word, we're going to use scikit-learn's [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

When you initialize TfidfVectorizer, you can choose to set it with different parameters. These parameters will change the way you calculate tf–idf.

The recommended way to run `TfidfVectorizer` is with smoothing (`smooth_idf = True`) and normalization (`norm='l2'`) turned on. These parameters will better account for differences in text length, and overall produce more meaningful tf–idf scores. Smoothing and L2 normalization are actually the default settings for `TfidfVectorizer`, so to turn them on, you don't need to include any extra code at all.

Initialize TfidfVectorizer with desired parameters (default smoothing and normalization)

Run TfidfVectorizer on our text

In [13]:
tfidf_vectorizer = TfidfVectorizer()
#tfidf_vectorizer = TfidfVectorizer(stop_words = 'english')
tfidf = tfidf_vectorizer.fit_transform(text).toarray()
tfidf

array([[0.        , 0.46979139, 0.58028582, 0.38408524, 0.        ,
        0.        , 0.38408524, 0.        , 0.38408524],
       [0.        , 0.6876236 , 0.        , 0.28108867, 0.        ,
        0.53864762, 0.28108867, 0.        , 0.28108867],
       [0.51184851, 0.        , 0.        , 0.26710379, 0.51184851,
        0.        , 0.26710379, 0.51184851, 0.26710379],
       [0.        , 0.46979139, 0.58028582, 0.38408524, 0.        ,
        0.        , 0.38408524, 0.        , 0.38408524]])

Get word and document labels for our word-document matrix

In [15]:
word_labels = tfidf_vectorizer.get_feature_names_out() 
doc_labels = ['D' + str(idx) for idx in range(len(text))]

print(word_labels)
print('\n')
print(doc_labels)

['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']


['D0', 'D1', 'D2', 'D3']


Make a DataFrame out of the resulting tf–idf vector, setting word labels as columns and document labels as rows

In [17]:
tfidf_df = pd.DataFrame(tfidf, index=doc_labels, columns=word_labels)

In [18]:
tfidf_df 

,and,document,first,is,one,second,the,third,this
D0,0.000000,0.469791,0.580286,0.384085,0.000000,0.000000,0.384085,0.000000,0.384085
D1,0.000000,0.687624,0.000000,0.281089,0.000000,0.538648,0.281089,0.000000,0.281089
D2,0.511849,0.000000,0.000000,0.267104,0.511849,0.000000,0.267104,0.511849,0.267104
D3,0.000000,0.469791,0.580286,0.384085,0.000000,0.000000,0.384085,0.000000,0.384085


Let's reorganize the DataFrame so that the words are in rows rather than columns.

In [20]:
tfidf_df = tfidf_df.stack().reset_index()
tfidf_df = tfidf_df.rename(columns={0:'tfidf', 'level_0': 'document','level_1': 'term', 'level_2': 'term'})

To find out the top 2 words with the highest tf–idf for every document, we're going to sort by document and tfidf score and then groupby document and take the first 10 values.

In [22]:
tfidf_df.sort_values(by=['document','tfidf'], ascending=[True,False]).groupby(['document']).head(2)

,document,term,tfidf
2,D0,first,0.580286
1,D0,document,0.469791
10,D1,document,0.687624
14,D1,second,0.538648
18,D2,and,0.511849
22,D2,one,0.511849
29,D3,first,0.580286
28,D3,document,0.469791


## TF-IDF with Gensim

[Gensim](https://radimrehurek.com/gensim/) is yet another popular libray specialising in statistical analysis of natural language text, in particular useful for Topic Modelling, which we will cover later in the unit.  

--edited upon the lab materials of [CITS4012](https://weiliu2k.github.io/CITS4012/index.html) 

In [24]:
import numpy as np
import gensim
from gensim import corpora, models, similarities

#### Step 1: Preprocessing
Gensim has a different routine in preparing text. It uses `gensim.utils.simple_preprocess()` to tokenise while removing punctuation and turn the tokens into lower cases. 

In [26]:
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True)) 

#### Step 2: Create a corpus with counts

Gensim has a built-in class `gensim.corpora.Dictionary` that has a function `doc2bow` that implements the bag of words idea, which processes the document collection, assigning an id to each unique token, while counting the term frequency of each token in each document. The following code returns each document as a list of tuples, in the form of `(term_id, count)`

In [108]:
doc_tokenized = list(sent_to_words(text))
dictionary = corpora.Dictionary()
BoW_corpus = [dictionary.doc2bow(doc, allow_update=True) for doc in doc_tokenized]
BoW_corpus
# print(dictionary)

[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)],
 [(0, 2), (2, 1), (3, 1), (4, 1), (5, 1)],
 [(2, 1), (3, 1), (4, 1), (6, 1), (7, 1), (8, 1)],
 [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)]]

We can examine the Bag of Words corpus obtained. You can think of the dictionary object is a vocabulary of the collection, mapping a term id to its lexical form (i.e. the word form) of the term. 

In [30]:
for doc in BoW_corpus:
   print([[dictionary[id], freq] for id, freq in doc])

[['document', 1], ['first', 1], ['is', 1], ['the', 1], ['this', 1]]
[['document', 2], ['is', 1], ['the', 1], ['this', 1], ['second', 1]]
[['is', 1], ['the', 1], ['this', 1], ['and', 1], ['one', 1], ['third', 1]]
[['document', 1], ['first', 1], ['is', 1], ['the', 1], ['this', 1]]


### Step 3: Calculating the tfidf values

A [`gensim.models.TfidfModel`](https://radimrehurek.com/gensim/models/tfidfmodel.html) object can be constructed using the processed BoW corpus. The `smartirs` parameter stands for SMART information retrieval system, where SMART is an acronym for "System for the Mechanical Analysis and Retrieval of Text". If interested, you can read more about [SMART on Wikipedia](https://en.wikipedia.org/wiki/SMART_Information_Retrieval_System), which contains a rather comprehensive list of TF-IDF variants.  `smartirs = ntc` means the model will use `n` raw term freqency, `t` zero-corrected idf, and `c` cosine for document vector nomalisation. For a list of other letter codes for each of the three tf-idf components, see the tabs or refer to the [original documentation](https://radimrehurek.com/gensim/models/tfidfmodel.html). You do not need to memorise all these variants, but just be aware of the many alternatives in calculating such a seemingly simple value. You can even define your own way of calculating tf and idf to feed into the model constructor. 

:::{tabbed} Term frequency weighing
- b - binary,
- t or n - raw,
- a - augmented,
- l - logarithm,
- d - double logarithm,
- L - log average.
:::

:::{tabbed} Document frequency weighting
- x or n - none,
- f - idf,
- t - zero-corrected idf,
- p - probabilistic idf.
::: 

:::{tabbed} Document normalization
- x or n - none,
- c - cosine,
- u - pivoted unique,
- b - pivoted character length.

In [32]:
tfidf = models.TfidfModel(BoW_corpus, smartirs='ntc')

# Get the tfidf vector representation of the second sentence
tfidf[BoW_corpus[1]]

[(0, 0.525241699949619),
 (2, 0.11472045720905524),
 (3, 0.11472045720905524),
 (4, 0.11472045720905524),
 (5, 0.8274290342544612)]

In [33]:
# Now a friendlier print out
for doc in tfidf[BoW_corpus]:
   print([[dictionary[id], np.around(freq,decimals=2)] for id, freq in doc])

[['document', 0.46], ['first', 0.82], ['is', 0.2], ['the', 0.2], ['this', 0.2]]
[['document', 0.53], ['is', 0.11], ['the', 0.11], ['this', 0.11], ['second', 0.83]]
[['is', 0.08], ['the', 0.08], ['this', 0.08], ['and', 0.57], ['one', 0.57], ['third', 0.57]]
[['document', 0.46], ['first', 0.82], ['is', 0.2], ['the', 0.2], ['this', 0.2]]


#### To get the document-term matrix

In [35]:
word_labels = [dictionary[i] for i in range(len(dictionary))]
word_labels

['document', 'first', 'is', 'the', 'this', 'second', 'and', 'one', 'third']

In [36]:
index = ['D'+str(idx) for idx in range(len(BoW_corpus))]
index

['D0', 'D1', 'D2', 'D3']

In [37]:
df = pd.DataFrame(data=np.zeros((len(BoW_corpus), len(word_labels)), dtype=np.float16),
                  index=index,
                  columns=word_labels)


In [38]:
for idx in range(len(index)):
    for id, freq in tfidf[BoW_corpus[idx]]:
        df[dictionary[id]][idx] = freq

C:\Users\leoxi\AppData\Local\Temp\ipykernel_1360\880531033.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[dictionary[id]][idx] = freq
C:\Users\leoxi\AppData\Local\Temp\ipykernel_1360\880531033.py:3: FutureWarning: Series.__setitem__ 

In [39]:
df

,document,first,is,the,this,second,and,one,third
D0,0.456913,0.819585,0.199593,0.199593,0.199593,0.000000,0.00000,0.00000,0.00000
D1,0.525242,0.000000,0.114720,0.114720,0.114720,0.827429,0.00000,0.00000,0.00000
D2,0.000000,0.000000,0.079289,0.079289,0.079289,0.000000,0.57188,0.57188,0.57188
D3,0.456913,0.819585,0.199593,0.199593,0.199593,0.000000,0.00000,0.00000,0.00000


## Vector Space Model

Let's calculate the cosine similarity of any two documents

In [41]:
from numpy.linalg import norm

def get_cosine_similarity(A, B):
    cosine = np.dot(A, B) / (norm(A) * norm(B))
    return cosine

In [42]:
sim_d0_d1 = get_cosine_similarity(df.loc['D0'].to_numpy(), df.loc['D1'].to_numpy())

sim_d0_d1

0.3086817099771662

## Latent Semantic Analysis

Let's continue working on gensim.

In [44]:
import os
import gensim
from gensim import corpora, models, similarities

#### Loading a folder of texts

In [46]:
def get_texts(fd_path):
    
    filenames = []
    texts = []

    for root, dirs, files in os.walk(fd_path):
        for file in files:
            if file.endswith('.txt'):
                
                f_path = os.path.join(root,file)
                f_txt = open(f_path, mode='r', encoding = 'utf-8').read()
                texts.append(f_txt)
                filenames.append(file.replace('.txt', ''))
    return texts, filenames

In [47]:
# foldr_path is the local directory path of folder "scistatistics_data" (in our wk2_lab)
folder_path = "../week 2/scistatistics_data"
texts, filenames = get_texts(folder_path)

In [48]:
texts[0:3]

['This paper presents an algorithm for labeling curvilinear structure at multiple scales in line drawings and edge images Symbolic CURVE-ELEMENT tokens residing in a spatially-indexed and scale-indexed data structure denote circular arcs fit to image data. Tokens are computed via a small-to-large scale grouping procedure employing a " greedy " , best-first, strategy for choosing the support of new tokens. The resulting image description is rich and redundant in that a given segment of image contour may be described by multiple tokens at different scales, and by more than one token at any given scale. This property facilitates selection and characterization of portions of the image based on local CURVE-ELEMENT attributes.\n',
 'The features based on Markov random field (MRF) models are usually sensitive to the rotation of image textures. This paper develops an anisotropic circular Gaussian MRF (ACGMRF) model for modelling rotated image textures and retrieving rotation-invariant texture 

In [49]:
filenames[0:3]

['CVPR_1992_10_abs', 'CVPR_2003_11_abs', 'CVPR_2003_18_abs']

In [50]:
import spacy
import en_core_web_sm

nlp = en_core_web_sm.load()

In [51]:
# texts = [t.split() for t in texts]
# texts[0]

corpus_tokens = []
for t in texts:
    doc = nlp(t)
    t_words = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and not token.is_space]
    corpus_tokens.append(t_words)

corpus_tokens[0]

['paper',
 'present',
 'algorithm',
 'label',
 'curvilinear',
 'structure',
 'multiple',
 'scale',
 'line',
 'drawing',
 'edge',
 'image',
 'Symbolic',
 'CURVE',
 'ELEMENT',
 'token',
 'reside',
 'spatially',
 'index',
 'scale',
 'index',
 'datum',
 'structure',
 'denote',
 'circular',
 'arc',
 'fit',
 'image',
 'datum',
 'Tokens',
 'compute',
 'small',
 'large',
 'scale',
 'grouping',
 'procedure',
 'employ',
 'greedy',
 'well',
 'strategy',
 'choose',
 'support',
 'new',
 'token',
 'result',
 'image',
 'description',
 'rich',
 'redundant',
 'give',
 'segment',
 'image',
 'contour',
 'describe',
 'multiple',
 'token',
 'different',
 'scale',
 'token',
 'give',
 'scale',
 'property',
 'facilitate',
 'selection',
 'characterization',
 'portion',
 'image',
 'base',
 'local',
 'CURVE',
 'ELEMENT',
 'attribute']

In [52]:
# Get the vocabulary
dictionary = corpora.Dictionary(corpus_tokens)

# Convert to a Bag-of-Words corpus
corpus = [dictionary.doc2bow(tokens) for tokens in corpus_tokens]

# Optional: Apply TF-IDF Transformation
tfidf = models.TfidfModel(corpus)
corpus_tfidf = tfidf[corpus]

# Perform LSA
lsi = models.LsiModel(corpus_tfidf, id2word=dictionary, num_topics=10)  # Adjust num_topics as needed, num_topics: number of vector dimensions 

In [53]:
# Get the document (topic) vectors
topic_word_matrix = lsi.get_topics() 

topic_word_matrix

array([[ 0.01755407,  0.01755407,  0.00877704, ...,  0.01596684,
         0.01596684,  0.01596684],
       [ 0.01503707,  0.01503707,  0.00751853, ..., -0.04437053,
        -0.04437053, -0.04437053],
       [-0.00115944, -0.00115944, -0.00057972, ..., -0.01412129,
        -0.01412129, -0.01412129],
       ...,
       [ 0.02836146,  0.02836146,  0.01418073, ...,  0.00168741,
         0.00168741,  0.00168741],
       [-0.01081528, -0.01081528, -0.00540764, ...,  0.02444642,
         0.02444642,  0.02444642],
       [ 0.01902727,  0.01902727,  0.00951364, ..., -0.0282436 ,
        -0.0282436 , -0.0282436 ]])

In [54]:
# Explore the topics
for topic in lsi.print_topics(num_topics=5):  # Adjust num_topics as needed
    print(topic)
    print('\n')

(0, '0.142*"object" + 0.136*"action" + 0.130*"robust" + 0.119*"model" + 0.114*"algorithm" + 0.112*"method" + 0.109*"training" + 0.104*"image" + 0.103*"system" + 0.102*"datum"')


(1, '0.198*"object" + -0.196*"system" + -0.170*"word" + -0.145*"grammar" + -0.121*"corpus" + -0.120*"unknown" + -0.117*"linguistic" + 0.113*"stereo" + 0.113*"motion" + -0.111*"lexicon"')


(2, '-0.527*"action" + -0.287*"attribute" + -0.257*"proposal" + 0.163*"robust" + -0.134*"agent" + -0.131*"video" + -0.104*"area" + -0.102*"learn" + -0.093*"instance" + -0.090*"motion"')


(3, '0.216*"classifier" + 0.191*"detection" + 0.176*"positive" + 0.173*"rate" + 0.171*"training" + 0.166*"example" + 0.160*"margin" + -0.122*"object" + -0.110*"system" + -0.108*"linguistic"')


(4, '-0.203*"action" + 0.180*"classifier" + -0.147*"matrix" + -0.147*"robust" + -0.143*"attribute" + -0.128*"PCA" + 0.127*"object" + -0.123*"exploit" + -0.123*"stochastic" + 0.118*"word"')




In [55]:
# Get the word vectors (term-topic matrix)
word_vectors = lsi.projection.u

# Get the vocabulary and create a mapping
# The index in the matrix corresponds to the word ID
word_labels = [dictionary[i] for i in range(len(dictionary))]

# Create a Pandas DataFrame for easier viewing/use
word_vectors_df = pd.DataFrame(word_vectors, index=word_labels)

word_vectors_df

,0,1,2,3,4,5,6,7,8,9
CURVE,0.017554,0.015037,-0.001159,0.009457,-0.024994,0.017263,-0.020542,0.028361,-0.010815,0.019027
ELEMENT,0.017554,0.015037,-0.001159,0.009457,-0.024994,0.017263,-0.020542,0.028361,-0.010815,0.019027
Symbolic,0.008777,0.007519,-0.000580,0.004729,-0.012497,0.008632,-0.010271,0.014181,-0.005408,0.009514
Tokens,0.008777,0.007519,-0.000580,0.004729,-0.012497,0.008632,-0.010271,0.014181,-0.005408,0.009514
algorithm,0.113972,0.044587,0.008610,0.012541,0.015576,-0.094452,-0.054853,0.069568,0.107269,-0.015606
...,...,...,...,...,...,...,...,...,...,...
cognitive,0.015967,-0.044371,-0.014121,-0.034877,0.007621,0.031420,0.000167,0.001687,0.024446,-0.028244
explore,0.015967,-0.044371,-0.014121,-0.034877,0.007621,0.031420,0.000167,0.001687,0.024446,-0.028244
prominence,0.015967,-0.044371,-0.014121,-0.034877,0.007621,0.031420,0.000167,0.001687,0.024446,-0.028244
psychology,0.015967,-0.044371,-0.014121,-0.034877,0.007621,0.031420,0.000167,0.001687,0.024446,-0.028244


### Question: Are the topics align with your expectations on the original data?

### Select any two relevant words that occur in the scientific text corpus, calculate their cosine similarity. Compare latent semantic analysis results with raw vector space model results

In [121]:
corpus_science = corpus_tokens[0]
corpus_science

['paper',
 'present',
 'algorithm',
 'label',
 'curvilinear',
 'structure',
 'multiple',
 'scale',
 'line',
 'drawing',
 'edge',
 'image',
 'Symbolic',
 'CURVE',
 'ELEMENT',
 'token',
 'reside',
 'spatially',
 'index',
 'scale',
 'index',
 'datum',
 'structure',
 'denote',
 'circular',
 'arc',
 'fit',
 'image',
 'datum',
 'Tokens',
 'compute',
 'small',
 'large',
 'scale',
 'grouping',
 'procedure',
 'employ',
 'greedy',
 'well',
 'strategy',
 'choose',
 'support',
 'new',
 'token',
 'result',
 'image',
 'description',
 'rich',
 'redundant',
 'give',
 'segment',
 'image',
 'contour',
 'describe',
 'multiple',
 'token',
 'different',
 'scale',
 'token',
 'give',
 'scale',
 'property',
 'facilitate',
 'selection',
 'characterization',
 'portion',
 'image',
 'base',
 'local',
 'CURVE',
 'ELEMENT',
 'attribute']